In [1]:
include("./common.jl")
using .Common
include("./factorbased.jl")
using .FactorBased
include("./factorizationconstruction.jl")
using .FactorizationConstruction

# BCRL Algorithm

This notebook implements an algorithm due to Bini, Capovani, Romani, and Lotti, who developed a factorization for the matrix-multiplication tensor with a rank that grows slightly more slowly than expected as the size of the matrix increases.  Specifically, they use a *border rank factorization*, in which, rather than factorizing the tensor as an element $K^{k \times m \times n}$, where $K$ is a field, they consider elements of $K[[\epsilon]]$ and, for some $h$, factorize the tensor multiplied by $\epsilon^h$ up to $O(\epsilon^{h+1})$.

Specifically, they prove that, when $h=1$, the border rank $R_1(\langle 2, 2, 3 \rangle)$ is at most $10$.  From this they determine that $R_{3s}(\langle 12^s, 12^s, 12^s \rangle) \leq 10^{3s}$, and hence that $$R(\langle 12^s, 12^s, 12^s \rangle) \leq (3s+1)(3s+2)(10^{3s})/2.$$  The exponent of matrix multiplication is therefore at most $$\log_{12^s}((3s+1)(3s+2)(10^{3s})/2) = 3 \log_{12}(10) + \frac{1}{s}\log_{12}((3s+1)(3s+2)/2).$$  It is at this point that we start to see why this algorithm is not used in practice: we first beat Strassen when $s=174$, or when dealing with matrices that have $12^{174} \approx 6 \times 10^{187}$ elements on each side.  [It has been estimated](https://doi.org/10.1063/5.0064475) that the observable universe as a whole stores approximately $6 \times 10^{80}$ *bits* of data.

## Step 1: Constructing a border rank factorization of $\langle 2, 2, 3 \rangle$

Consider the map that multiplies $2 \times 3$ matrices but only computes the three top-left elements, thus: $$\begin{pmatrix}
x_{11} & x_{12} \\ x_{21} & x_{22}
\end{pmatrix} \begin{pmatrix}
y_{11} & y_{12} & y_{13} \\ y_{21} & y_{22} & y_{23}
\end{pmatrix} = \begin{pmatrix}
z_{11} & z_{12} & \cdot \\ z_{21} & \cdot & \cdot
\end{pmatrix}.$$  If we apply this map twice, but reflecting the first matrix vertically and the second horizontally the second time, we can get all six entries of the remaining matrix.  We will first build a factorization of this map, then use it to factor $\langle 2, 2, 3 \rangle$.

In [2]:
alphas_constant = [
    0 1; 0 0;;;
    1 0; 0 0;;;
    0 1; 0 0;;;
    1 1; 0 0;;;
    0 1; 0 0;;;
    0 0; 0 1;;;
    0 0; 1 0;;;
    0 0; 0 1;;;
    0 0; 1 1;;;
    0 0; 0 1;;;
]

alphas_epsilon = [
    0 0; 0 1;;;
    0 0; 0 0;;;
    0 0; 0 0;;;
    0 0; 1 0;;;
    0 0; 1 0;;;
    0 1; 0 0;;;
    0 0; 0 0;;;
    0 0; 0 0;;;
    1 0; 0 0;;;
    1 0; 0 0;;;
]

alphas = collect(zip(eachslice(alphas_constant, dims=3), eachslice(alphas_epsilon, dims=3)))

betas_constant = [
    0 0 0; 0 1 0;;;
    1 0 0; 0 0 0;;;
    1 0 0; 1 0 0;;;
    1 0 0; 0 0 0;;;
    1 0 0; 0 0 0;;;
    0 0 0; 0 1 0;;;
    0 0 1; 0 0 0;;;
    0 0 1; 0 0 1;;;
    0 0 1; 0 0 0;;;
    0 0 1; 0 0 0;;;
]

betas_epsilon = [
    0 0 0; 0 0 0;;;
    0 1 0; 0 0 0;;;
    0 0 0; 0 1 0;;;
    0 0 0; 0 0 0;;;
    0 0 0; 0 1 0;;;
    0 0 0; 0 0 0;;;
    0 1 0; 0 0 0;;;
    0 0 0; 0 1 0;;;
    0 0 0; 0 0 0;;;
    0 0 0; 0 1 0;;;
]

betas = collect(zip(eachslice(betas_constant, dims=3), eachslice(betas_epsilon, dims=3)))

gammas_constant = [
    0 0 0; 1 0 0;;;
    0 1 0; 0 0 0;;;
    0 0 0; 0 -1 0;;;
    0 -1 0; 0 0 0;;;
    0 1 0; 1 0 0;;;
    0 0 1; 0 0 0;;;
    0 0 0; 0 1 0;;;
    0 0 -1; 0 0 0;;;
    0 0 0; 0 -1 0;;;
    0 0 1; 0 1 0;;;
]

gammas_epsilon = [
    1 0 0; 0 0 0;;;
    1 0 0; 0 0 0;;;
    0 0 0; 0 0 0;;;
    0 0 0; 0 0 0;;;
    0 0 0; 0 0 0;;;
    0 0 0; 0 0 1;;;
    0 0 0; 0 0 1;;;
    0 0 0; 0 0 0;;;
    0 0 0; 0 0 0;;;
    0 0 0; 0 0 0;;;
]

gammas = collect(zip(eachslice(gammas_constant, dims=3), eachslice(gammas_epsilon, dims=3)))

;

## Step 2: Transforming this into a factorization of $\langle 12^s, 12^s, 12^s \rangle$

In [3]:
alphas, betas, gammas = square_up(alphas, betas, gammas)

([[0], [0], [0], [0], [0], [0], [0], [0], [0], [0]  …  [0], [0], [0], [0], [0], [0], [0], [0], [0], [0]], [[0], [0], [0], [0], [0], [0], [0], [0], [0], [0]  …  [0], [0], [0], [0], [0], [0], [0], [0], [0], [0]], [[0], [0], [0], [0], [0], [0], [0], [0], [0], [0]  …  [0], [0], [0], [0], [0], [0], [0], [0], [0], [0]])

In [7]:
sum(gammas)

1-element Vector{Int64}:
 0